In [12]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports/evaluation_w14")

TRAIN_DATASET_PATH = (
    PROCESSED_DIR / "intersection_day_2018_features.csv"
)

EVENTS_PATH = (
    PROCESSED_DIR / "ksi_matched_events.csv"
)

WEATHER_PATH = (
    RAW_DIR / "en_climate_daily_ON_6158731_2019_P1D.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR / "intersection_day_2019_features.csv"
)

with (REPORTS_DIR / "model_selection.json").open(
    encoding="utf-8"
) as file:
    model_selection = json.load(file)

TEST_START = pd.Timestamp(
    model_selection["final_test"]["start"]
)

TEST_END = pd.Timestamp(
    model_selection["final_test"]["end"]
)

feature_columns = model_selection["feature_columns"]

print(f"Test period: {TEST_START.date()} — {TEST_END.date()}")
print(f"Required features: {len(feature_columns)}")

Test period: 2019-01-02 — 2019-12-30
Required features: 11


In [13]:
reference_df = pd.read_csv(
    TRAIN_DATASET_PATH,
    parse_dates=[
        "prediction_date",
        "target_date",
        "weather_date"
    ]
)

matched_events = pd.read_csv(
    EVENTS_PATH,
    usecols=[
        "intersection_id",
        "collision_id",
        "event_date"
    ],
    parse_dates=["event_date"]
)

weather_raw = pd.read_csv(
    WEATHER_PATH,
    encoding="utf-8-sig",
    low_memory=False
)

assert weather_raw["Climate ID"].eq(6158731).all()
assert weather_raw["Station Name"].eq("TORONTO INTL A").all()

weather_dates = pd.to_datetime(
    weather_raw["Date/Time"]
).dt.normalize()

required_weather_dates = pd.date_range(
    TEST_START - pd.Timedelta(days=1),
    TEST_END - pd.Timedelta(days=1),
    freq="D"
)

assert required_weather_dates.isin(weather_dates).all()

print("Reference dataset:", reference_df.shape)
print("Weather dataset:", weather_raw.shape)
print("Weather station and date coverage verified.")

Reference dataset: (823217, 16)
Weather dataset: (365, 31)
Weather station and date coverage verified.


In [14]:
intersection_ids = np.sort(
    reference_df["intersection_id"].unique()
)

prediction_dates = pd.date_range(
    start=TEST_START,
    end=TEST_END,
    freq="D"
)

test_df = pd.MultiIndex.from_product(
    [intersection_ids, prediction_dates],
    names=[
        "intersection_id",
        "prediction_date"
    ]
).to_frame(index=False)

test_df["target_date"] = (
    test_df["prediction_date"]
    + pd.Timedelta(days=1)
)

print(f"Intersections: {len(intersection_ids):,}")
print(f"Prediction days: {len(prediction_dates)}")
print(f"Test observations: {len(test_df):,}")

Intersections: 2,258
Prediction days: 363
Test observations: 819,654


In [15]:
test_df["day_of_week"] = (
    test_df["target_date"].dt.dayofweek
)

test_df["month"] = (
    test_df["target_date"].dt.month
)

test_df["is_weekend"] = (
    test_df["day_of_week"]
    .isin([5, 6])
    .astype("int8")
)

traffic_variability = (
    reference_df
    .groupby("intersection_id")[
        "historical_traffic_volume"
    ]
    .nunique(dropna=False)
)

assert traffic_variability.le(1).all()

traffic_reference = (
    reference_df[[
        "intersection_id",
        "historical_traffic_volume"
    ]]
    .drop_duplicates()
)

test_df = test_df.merge(
    traffic_reference,
    on="intersection_id",
    how="left",
    validate="many_to_one"
)

test_df["traffic_missing"] = (
    test_df["historical_traffic_volume"]
    .isna()
    .astype("int8")
)

print("Calendar and traffic features created.")

Calendar and traffic features created.


In [16]:
matched_events = (
    matched_events
    .drop_duplicates([
        "intersection_id",
        "collision_id",
        "event_date"
    ])
    .copy()
)

event_history = {
    intersection_id: np.sort(
        group["event_date"].to_numpy(
            dtype="datetime64[D]"
        )
    )
    for intersection_id, group in matched_events.groupby(
        "intersection_id"
    )
}

history_columns = [
    "historical_ksi_count",
    "ksi_count_30d",
    "ksi_count_365d"
]


def build_history_features(frame):
    dates = frame["prediction_date"].to_numpy(
        dtype="datetime64[D]"
    )

    counts = np.zeros(
        (len(frame), len(history_columns)),
        dtype=np.int64
    )

    for intersection_id, positions in frame.groupby(
        "intersection_id"
    ).indices.items():

        history = event_history.get(intersection_id)

        if history is None:
            continue

        prediction_dates = dates[positions]

        right = np.searchsorted(
            history,
            prediction_dates,
            side="left"
        )

        left_30 = np.searchsorted(
            history,
            prediction_dates - np.timedelta64(30, "D"),
            side="left"
        )

        left_365 = np.searchsorted(
            history,
            prediction_dates - np.timedelta64(365, "D"),
            side="left"
        )

        counts[positions, 0] = right
        counts[positions, 1] = right - left_30
        counts[positions, 2] = right - left_365

    return counts


test_df[history_columns] = build_history_features(test_df)

print("Historical KSI features created.")

Historical KSI features created.


In [17]:
weather = weather_raw[[
    "Date/Time",
    "Mean Temp (°C)",
    "Total Precip (mm)"
]].copy()

weather = weather.rename(columns={
    "Date/Time": "weather_date",
    "Mean Temp (°C)": "temperature_prev_day",
    "Total Precip (mm)": "precipitation_prev_day"
})

weather["weather_date"] = pd.to_datetime(
    weather["weather_date"]
).dt.normalize()

for col in [
    "temperature_prev_day",
    "precipitation_prev_day"
]:
    weather[col] = pd.to_numeric(
        weather[col],
        errors="coerce"
    )

assert weather["weather_date"].is_unique

test_df["weather_date"] = (
    test_df["prediction_date"]
    - pd.Timedelta(days=1)
)

test_df = test_df.merge(
    weather,
    on="weather_date",
    how="left",
    validate="many_to_one"
)

test_df["weather_missing"] = (
    test_df[[
        "temperature_prev_day",
        "precipitation_prev_day"
    ]]
    .isna()
    .any(axis=1)
    .astype("int8")
)

print("Weather features created.")

Weather features created.


In [18]:
daily_ksi = (
    matched_events
    .groupby([
        "intersection_id",
        "event_date"
    ])["collision_id"]
    .nunique()
    .reset_index(name="next_day_ksi")
)

next_day_events = daily_ksi.rename(
    columns={
        "event_date": "target_date"
    }
)

test_df = test_df.merge(
    next_day_events,
    on=[
        "intersection_id",
        "target_date"
    ],
    how="left",
    validate="one_to_one"
)

test_df["target"] = (
    test_df["next_day_ksi"]
    .fillna(0)
    .gt(0)
    .astype("int8")
)

test_df = test_df.drop(
    columns=["next_day_ksi"]
)

print("Next-day target created.")

Next-day target created.


In [19]:
expected_rows = (
    len(intersection_ids) * len(prediction_dates)
)

assert len(test_df) == expected_rows

assert not test_df.duplicated([
    "intersection_id",
    "prediction_date"
]).any()

assert (
    test_df["target_date"]
    - test_df["prediction_date"]
).eq(pd.Timedelta(days=1)).all()

assert test_df["target"].isin([0, 1]).all()

assert set(test_df.columns) == set(reference_df.columns)

assert set(feature_columns).issubset(test_df.columns)

reference_history = build_history_features(
    reference_df
)

assert np.array_equal(
    reference_history,
    reference_df[history_columns].to_numpy()
)

test_df = (
    test_df[reference_df.columns]
    .sort_values([
        "prediction_date",
        "intersection_id"
    ])
    .reset_index(drop=True)
)

test_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("All dataset integrity checks passed.")
print(f"Test dataset shape: {test_df.shape}")
print(f"Saved to: {OUTPUT_PATH}")

All dataset integrity checks passed.
Test dataset shape: (819654, 16)
Saved to: ..\data\processed\intersection_day_2019_features.csv
